In [7]:
import pandas as pd
from lightgbm import LGBMClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import StandardScaler
from utils import *
import numpy as np
import wandb
from utils.config import SEED


In [3]:
df = pd.read_csv('../../../data/creditcard.csv')

df = create_features(df)


df['hour_sin'] = np.sin(2 * np.pi * df['Hour_from_start_mod24']/24)
df['hour_cos'] = np.cos(2 * np.pi * df['Hour_from_start_mod24']/24)
df = df.sort_values('Time')
df['time_diff'] = df['Time'].diff().fillna(0)
threshold = df['Amount'].quantile(0.95)  
df['is_high_amount'] = (df['Amount'] > threshold).astype(int)
df['is_rapid_transaction'] = (df['time_diff'] < 60).astype(int)

df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,Class,_log_amount,Hour_from_start_mod24,is_night_proxy,is_business_hours_proxy,hour_sin,hour_cos,time_diff,is_high_amount,is_rapid_transaction
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0,5.014760,0,1,0,0.0,1.0,0.0,0,1
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,0,1.305626,0,1,0,0.0,1.0,0.0,0,1
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0,5.939276,0,1,0,0.0,1.0,1.0,1,1
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,0,4.824306,0,1,0,0.0,1.0,0.0,0,1
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,0,4.262539,0,1,0,0.0,1.0,1.0,0,1


In [4]:
features = [c for c in df.columns if c.startswith("V")] + [
    'Amount','_log_amount',
    'Hour_from_start_mod24','is_night_proxy','is_business_hours_proxy',
    'hour_sin','hour_cos','time_diff','is_high_amount','is_rapid_transaction'
]
target = "Class"

In [5]:
X_train, y_train, X_val, y_val, X_test, y_test = split_data(df, features, target)

X_train: (181584, 38) y_train: (181584,)
X_val: (45396, 38) y_val: (45396,)
X_test: (56746, 38) y_test: (56746,)
Fraud rate in train: 0.001910961318177813
Fraud rate in test: 0.0013040566735981391


In [18]:
run = start_run(
    model_name="LightGBM",
    config={
        "n_estimators":1200,
        "learning_rate":0.02,
        "num_leaves":64,
        "min_child_samples":20,
        "max_depth":-1,
        "lambda_l2":1.0,
    }
)


wandb: WARNING Unable to render HTML, can't import display from ipython.core
c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2265: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stderr", data),


wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2265: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stderr", data),
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't im

In [21]:

lgbm_pipe = ImbPipeline(steps=[
    ("model", LGBMClassifier(
        n_estimators=wandb.config.n_estimators,
        learning_rate=wandb.config.learning_rate,
        num_leaves=wandb.config.num_leaves,
        min_child_samples=wandb.config.min_child_samples,
        max_depth=wandb.config.max_depth,
        lambda_l2=wandb.config.lambda_l2,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=SEED,
        class_weight="balanced",
        eval_metric="average_precision"
    ))
])

lgbm_pipe.fit(X_train, y_train)

val_lgb_proba  = lgbm_pipe.predict_proba(X_val)[:,1]
test_lgb_proba = lgbm_pipe.predict_proba(X_test)[:,1]

[LightGBM] [Warning] Unknown parameter: eval_metric

c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2259: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stdout", data),
c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2265: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stderr", data),



[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0
[LightGBM] [Info] Number of positive: 347, number of negative: 181237
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011925 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7746
[LightGBM] [Info] Number of data points in the train set: 181584, number of used features: 37
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning]

c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2259: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stdout", data),
c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2265: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stderr", data),



[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0
[LightGBM] [Warning] Unknown parameter: eval_metric

c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2259: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stdout", data),
c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2265: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stderr", data),



[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0


In [22]:
result_lgb_val = log_eval(y_val, val_lgb_proba)
result_lgb_test = log_eval(y_test, test_lgb_proba)
print("LightGBM - Validation Set:")
print(result_lgb_val)
print("LightGBM - Test Set:")
print(result_lgb_test)

LightGBM - Validation Set:

c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2259: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stdout", data),
c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2265: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stderr", data),



{'threshold': 0.001, 'Cost': 2370.0, 'ROC_AUC': 0.9759178553010152, 'PR_AUC': 0.7847698833394652, 'debiased_ece': 0.000402182378347557, 'adaptive_ece': 0.0001240415200975296, 'Brier': 0.0004009370275583528}
LightGBM - Test Set:
{'threshold': 0.021, 'Cost': 3125.0, 'ROC_AUC': 0.9804591523341524, 'PR_AUC': 0.7875096268879426, 'debiased_ece': 0.00043103586557166694, 'adaptive_ece': 0.00016456395519734532, 'Brier': 0.00044395100920992253}


In [27]:
result = evaluate(y_test, test_lgb_proba)
print(f"TP={result['tp']}, FP={result['fp']}, FN={result['fn']}, TN={result['tn']}")
print(pd.Series(y_test).value_counts())

TP=56, FP=7, FN=18, TN=56665

c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2259: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stdout", data),
c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2265: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stderr", data),



Class
0    56672
1       74
Name: count, dtype: int64

c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2259: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stdout", data),
c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2265: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stderr", data),


In [23]:
wandb.log(result_lgb_test)
wandb.log(result_lgb_val)
wandb.finish()

wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core


### Calibrated

In [32]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

In [30]:
cal_cv = TimeSeriesSplit(n_splits=3)

In [33]:
lgbm_cal = CalibratedClassifierCV(estimator = lgbm_pipe, method='isotonic', cv=cal_cv)
lgbm_cal.fit(X_train, y_train)
val_lgb_cal_proba = lgbm_cal.predict_proba(X_val)[:,1]
test_lgb_cal_proba = lgbm_cal.predict_proba(X_test)[:,1]

[LightGBM] [Warning] Unknown parameter: eval_metric

c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2259: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stdout", data),
c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2265: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stderr", data),



[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0
[LightGBM] [Info] Number of positive: 142, number of negative: 45254
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004329 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7713
[LightGBM] [Info] Number of data points in the train set: 45396, number of used features: 37
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No 

c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2259: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stdout", data),
c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2265: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stderr", data),



[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0
[LightGBM] [Warning] Unknown parameter: eval_metric

...


[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0
[LightGBM] [Warning] Unknown parameter: eval_metric

c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2259: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stdout", data),
c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2265: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stderr", data),



[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0
[LightGBM] [Warning] Unknown parameter: eval_metric

c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2259: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stdout", data),
c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2265: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stderr", data),



[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0


In [34]:
result_lgbm_cal_val = log_eval(y_val, val_lgb_cal_proba)
result_lgbm_cal_test = log_eval(y_test, test_lgb_cal_proba)
print("Calibrated LightGBM - Validation Set:")
print(result_lgbm_cal_val)
print("Calibrated LightGBM - Test Set:")
print(result_lgbm_cal_test)

Calibrated LightGBM - Validation Set:

c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2259: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stdout", data),
c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2265: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stderr", data),



{'threshold': 0.099, 'Cost': 2315.0, 'ROC_AUC': 0.9587639446826991, 'PR_AUC': 0.7540164424673178, 'debiased_ece': 0.0006223212841738555, 'adaptive_ece': 0.0006793215154306386, 'Brier': 0.00052294194699046}
Calibrated LightGBM - Test Set:
{'threshold': 0.095, 'Cost': 3240.0, 'ROC_AUC': 0.9717808355715964, 'PR_AUC': 0.7717925574086641, 'debiased_ece': 0.0005266513352456898, 'adaptive_ece': 0.0004967155156568072, 'Brier': 0.00052094114482282}


In [35]:
result_cal = evaluate(y_test, test_lgb_cal_proba)
print(f"TP={result_cal['tp']}, FP={result_cal['fp']}, FN={result_cal['fn']}, TN={result_cal['tn']}")
print(pd.Series(y_test).value_counts())

TP=57, FP=17, FN=17, TN=56655

c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2259: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stdout", data),
c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2265: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stderr", data),



Class
0    56672
1       74
Name: count, dtype: int64

c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2259: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stdout", data),
c:\Users\huyy\Documents\Credit-Card-Fraud-Detection-Classification---Imbalanced-\.venv\Lib\site-packages\wandb\sdk\wandb_run.py:2265: UserWarning: Run (x24igweh) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stderr", data),
